# Task 3 — Hindi → English Translation of 5-sentence groups (Kaggle)

**Input:** `all_groups.json` from Task 2 (ordered sentences, grouped in 5s)
**Model:** `ai4bharat/indictrans2-indic-en-dist-200M` (pre-trained, ~200M params, used as-is, no fine-tuning)
**Pipeline:** IndicProcessor preprocessing → tokenizer → beam search (5 beams) → IndicProcessor postprocessing
**Safety checks:** each group is translated as one unit when it fits the model's 256-token limit; longer groups are split at sentence boundaries; outputs that hit the length limit are re-translated sentence by sentence; leftover Devanagari in the English is flagged.
**GPU memory:** hard-capped at 1.45 GB.

### Setup
1. **Hugging Face token** (the model is gated): see the steps in the chat. Add it as a Kaggle secret named `HF_TOKEN` (*Add-ons → Secrets*) and make sure it's **attached** to this notebook.
2. **Add Input → Your Work → Notebooks →** your Task 2 notebook (the latest saved version).
3. **Accelerator → GPU T4 x2**, **Internet → On**.
4. Run cells one by one. **Run CELL 1 first, before anything else** (it installs the right library versions).

In [ ]:
# ===== CELL 1: Install the versions IndicTrans2 needs (run FIRST, in a fresh session) =====
# IndicTrans2's custom model code is written for transformers 4.x, so we pin <5 in THIS notebook only.
!pip install -q "transformers>=4.51,<5" indictranstoolkit sacrebleu

In [ ]:
# ===== CELL 2: Imports, config, GPU memory cap, Hugging Face login =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, json, glob, zipfile, shutil, time, collections, contextlib, warnings, math
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import torch
import transformers

assert int(transformers.__version__.split(".")[0]) < 5, (
    f"transformers {transformers.__version__} is loaded. Run CELL 1, then Run > Restart & clear outputs, then start from CELL 2.")

MODEL_ID    = "ai4bharat/indictrans2-indic-en-dist-200M"
SRC_LANG    = "hin_Deva"
TGT_LANG    = "eng_Latn"
MAX_LEN     = 256     # model's maximum sequence length
PACK_LIMIT  = 200     # max source tokens per translation unit (leaves room so the English output fits in 256)
NUM_BEAMS   = 5
BATCH_SIZE  = 8
SEED        = 42
WORK        = "/kaggle/working"
OUT_DIR     = f"{WORK}/outputs"
for d in [OUT_DIR, f"{OUT_DIR}/english", f"{OUT_DIR}/parallel"]:
    os.makedirs(d, exist_ok=True)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    _total = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.set_per_process_memory_fraction(min(1.0, 1.45 * 1024**3 / _total), 0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | memory hard-capped at 1.45 GB")
else:
    print("WARNING: no GPU. Turn on GPU T4 x2 (CPU works but is slow).")

from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception as e:
    raise RuntimeError("HF_TOKEN secret not found. Add-ons -> Secrets -> add HF_TOKEN and tick the checkbox to attach it.") from e

print("transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
# ===== CELL 3: Load the ordered groups from Task 2 =====
found = glob.glob("/kaggle/input/**/all_groups.json", recursive=True)
if not found:
    for z in glob.glob("/kaggle/input/**/task2_outputs.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(f"{WORK}/task2_outputs")
    found = glob.glob(f"{WORK}/task2_outputs/**/all_groups.json", recursive=True)
assert found, "all_groups.json not found. Add your Task 2 notebook (saved version) as input."
print("Using:", found[0])

def sample_key(name):
    m = re.search(r"sample-(\d+)", name)
    return int(m.group(1)) if m else 999

ALL_GROUPS = json.load(open(found[0], encoding="utf-8"))
ALL_GROUPS = {k: ALL_GROUPS[k] for k in sorted(ALL_GROUPS, key=sample_key)}

for name, groups in ALL_GROUPS.items():
    assert all(isinstance(g, list) and g for g in groups), f"bad group format in {name}"
    print(f"{name}: {len(groups)} groups, {sum(len(g) for g in groups)} sentences")

In [ ]:
# ===== CELL 4: Load IndicTrans2 + processor, then a sanity test =====
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
try:
    from IndicTransToolkit import IndicProcessor
except ImportError:
    from IndicTransToolkit.IndicTransToolkit import IndicProcessor

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=True)
model = (model.half() if device == "cuda" else model).to(device).eval()

# IMPORTANT: IndicProcessor remembers placeholders between preprocess and postprocess.
# 'ip' is used ONLY for translation (preprocess -> generate -> postprocess, in that order).
# 'ip_count' is a separate instance used only for counting tokens, so it can never mix them up.
ip       = IndicProcessor(inference=True)
ip_count = IndicProcessor(inference=True)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.1f}M")

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else model.config.pad_token_id
USE_CACHE = True

def _generate(inputs, use_cache):
    with torch.inference_mode():
        return model.generate(**inputs, num_beams=NUM_BEAMS, num_return_sequences=1,
                              max_length=MAX_LEN, min_length=0, use_cache=use_cache)

def translate_texts(texts):
    # Translate a list of Hindi strings. Returns (english_list, hit_length_limit_list).
    global USE_CACHE
    out_en, out_hit = [], []
    for i in range(0, len(texts), BATCH_SIZE):
        chunk = texts[i:i + BATCH_SIZE]
        batch = ip.preprocess_batch(chunk, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
        inputs = tokenizer(batch, truncation=True, padding="longest", max_length=MAX_LEN,
                           return_tensors="pt", return_attention_mask=True).to(device)
        try:
            gen = _generate(inputs, USE_CACHE)
        except Exception as e:
            if not USE_CACHE:
                raise
            print("generate() with use_cache=True failed -> switching to use_cache=False:", repr(e)[:200])
            USE_CACHE = False
            gen = _generate(inputs, USE_CACHE)
        lengths = (gen != PAD_ID).sum(dim=1).tolist()
        # decode with the TARGET (English) vocabulary; safe on old and new tokenizer versions
        ctx = tokenizer.as_target_tokenizer() if hasattr(tokenizer, "as_target_tokenizer") else contextlib.nullcontext()
        with ctx:
            decoded = tokenizer.batch_decode(gen.detach().cpu().tolist(), skip_special_tokens=True,
                                             clean_up_tokenization_spaces=True)
        en = ip.postprocess_batch(decoded, lang=TGT_LANG)
        out_en += [e.strip() for e in en]
        out_hit += [l >= MAX_LEN - 1 for l in lengths]
    return out_en, out_hit

def n_src_tokens(text):
    b = ip_count.preprocess_batch([text], src_lang=SRC_LANG, tgt_lang=TGT_LANG)
    return len(tokenizer(b, truncation=False)["input_ids"][0])

DEVA_RE = re.compile("[\u0900-\u097F]")
test = ["जब मैं छोटा था, मैं हर रोज़ पार्क जाता था।", "गंगा भारत की सबसे पवित्र नदी मानी जाती है।"]
en, hit = translate_texts(test)
for h, e in zip(test, en):
    print(f"HI: {h}\nEN: {e}\n")
ok = all(e and not DEVA_RE.search(e) and len(re.findall(r"[A-Za-z]+", e)) >= 3 for e in en)
assert ok, "Sanity test failed: output is empty, still Hindi, or not English. Send this cell's output."
print("Sanity test passed | use_cache =", USE_CACHE)

In [ ]:
# ===== CELL 5: Build translation units (1 group = 1 unit unless it is too long) =====
def pack(sents, limit=PACK_LIMIT):
    # Split a group into consecutive pieces whose source length is <= limit tokens.
    pieces, cur = [], []
    for s in sents:
        if cur and n_src_tokens(" ".join(cur + [s])) > limit:
            pieces.append(cur); cur = [s]
        else:
            cur = cur + [s]
    if cur:
        pieces.append(cur)
    return pieces

units = []          # each unit: {"sample", "group", "piece", "sents"}
split_groups, long_sentences = 0, 0
for name, groups in ALL_GROUPS.items():
    for gi, g in enumerate(groups):
        pieces = pack(g)
        if len(pieces) > 1:
            split_groups += 1
        for pi, p in enumerate(pieces):
            if len(p) == 1 and n_src_tokens(p[0]) > MAX_LEN:
                long_sentences += 1
            units.append({"sample": name, "group": gi, "piece": pi, "sents": p})

n_groups = sum(len(g) for g in ALL_GROUPS.values())
print(f"Groups: {n_groups} | translation units: {len(units)} | groups that had to be split: {split_groups}")
print(f"Single sentences longer than {MAX_LEN} tokens (would be cut): {long_sentences}")

In [ ]:
# ===== CELL 6: Translate everything (with automatic re-translation of cut-off outputs) =====
if device == "cuda":
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()

texts = [" ".join(u["sents"]) for u in units]
english, hit = translate_texts(texts)
for u, e, h in zip(units, english, hit):
    u["en"], u["hit_limit"] = e, h

# Any multi-sentence unit whose output hit the length limit -> translate its sentences one by one.
retried = 0
for u in units:
    if u["hit_limit"] and len(u["sents"]) > 1:
        en_s, hit_s = translate_texts(u["sents"])
        u["en"], u["hit_limit"], u["retranslated"] = " ".join(en_s), any(hit_s), True
        retried += 1

# Any multi-sentence unit whose English length looks wrong (model may have skipped text)
# -> translate its sentences one by one and keep whichever version has the more natural length.
def wratio(hi, en):
    return len(en.split()) / max(1, len(hi.split()))
ratio_fixed = []
for u in units:
    hi = " ".join(u["sents"]); r = wratio(hi, u["en"])
    if (r < 0.5 or r > 2.0) and len(u["sents"]) > 1:
        en_s, hit_s = translate_texts(u["sents"])
        alt = " ".join(en_s); r2 = wratio(hi, alt)
        if abs(math.log(max(r2, 1e-3))) < abs(math.log(max(r, 1e-3))):
            u["en"], u["hit_limit"] = alt, any(hit_s)
            ratio_fixed.append({"sample": u["sample"], "group": u["group"] + 1,
                                "ratio_before": round(r, 2), "ratio_after": round(r2, 2)})
print("Re-translated because of unusual length:", ratio_fixed if ratio_fixed else "none")

elapsed = time.time() - t0
infer_peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0.0
infer_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3 if device == "cuda" else 0.0
print(f"Translated {len(units)} units in {elapsed:.0f}s | re-translated sentence-by-sentence: {retried}")
print(f"Still at length limit after retry: {sum(u['hit_limit'] for u in units)}")
print(f"Peak GPU memory: {infer_peak_gb:.3f} GB allocated ({infer_reserved_gb:.3f} GB reserved)")
print("UNDER 1.5 GB" if infer_reserved_gb < 1.5 else "OVER 1.5 GB")

In [ ]:
# ===== CELL 7: Assemble per group, write files, quality checks =====
DEVANAGARI = re.compile(r"[ऀ-ॿ]")
results = collections.OrderedDict()
for name, groups in ALL_GROUPS.items():
    results[name] = [{"group": gi + 1, "hindi": " ".join(g), "english": ""} for gi, g in enumerate(groups)]
for u in units:
    r = results[u["sample"]][u["group"]]
    r["english"] = (r["english"] + " " + u["en"]).strip()

issues = []
for name, rows in results.items():
    for r in rows:
        ratio = len(r["english"].split()) / max(1, len(r["hindi"].split()))
        r["en_hi_word_ratio"] = round(ratio, 2)
        if not r["english"]:
            issues.append((name, r["group"], "EMPTY"))
        elif DEVANAGARI.search(r["english"]):
            issues.append((name, r["group"], "Hindi characters left in English"))
        elif ratio < 0.5 or ratio > 2.0:
            issues.append((name, r["group"], f"unusual length ratio {ratio:.2f}"))

    with open(f"{OUT_DIR}/english/{name}", "w", encoding="utf-8") as f:
        f.write("\n\n".join(f"### Group {r['group']}\n{r['english']}" for r in rows))
    with open(f"{OUT_DIR}/parallel/{name.replace('.txt', '.md')}", "w", encoding="utf-8") as f:
        f.write(f"# {name}\n\n" + "\n\n".join(
            f"## Group {r['group']}\n\n**Hindi:** {r['hindi']}\n\n**English:** {r['english']}" for r in rows))

json.dump(results, open(f"{OUT_DIR}/all_translations.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

import csv
with open(f"{OUT_DIR}/all_translations.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f); w.writerow(["sample", "group", "hindi", "english"])
    for name, rows in results.items():
        for r in rows:
            w.writerow([name, r["group"], r["hindi"], r["english"]])

total_groups = sum(len(v) for v in results.values())
print(f"{total_groups} groups translated | issues flagged: {len(issues)}")
for it in issues[:20]:
    print("  ", it)

name = list(results)[0]
for r in results[name][:2]:
    print(f"\n--- {name} | Group {r['group']} ---\nHI: {r['hindi'][:300]}\nEN: {r['english'][:300]}")

In [ ]:
# ===== CELL 8 (optional): Measure the model's quality on FLORES-200 Hindi->English =====
# Needs access to the gated dataset openlanguagedata/flores_plus (accept its terms on Hugging Face with the
# same account). If it is not accessible, this cell just skips; nothing else depends on it.
flores = None
try:
    from datasets import load_dataset
    import sacrebleu
    hi_ds = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split="devtest")
    en_ds = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
    en_by_id = {r["id"]: r["text"] for r in en_ds}
    pairs = [(r["text"], en_by_id[r["id"]]) for r in hi_ds if r["id"] in en_by_id][:200]
    hyp, _ = translate_texts([p[0] for p in pairs])
    refs = [p[1] for p in pairs]
    flores = {"sentences": len(pairs),
              "BLEU": round(sacrebleu.corpus_bleu(hyp, [refs]).score, 2),
              "chrF++": round(sacrebleu.corpus_chrf(hyp, [refs], word_order=2).score, 2)}
    print("FLORES-200 devtest (hin_Deva -> eng_Latn):", flores)
except Exception as e:
    print("FLORES check skipped:", repr(e)[:200])

In [ ]:
# ===== CELL 9: Report + zip =====
from IPython.display import FileLink, display
report = {
    "model": MODEL_ID, "params_M": round(n_params / 1e6, 1), "pretrained_only": True,
    "src_lang": SRC_LANG, "tgt_lang": TGT_LANG, "num_beams": NUM_BEAMS, "max_len": MAX_LEN, "pack_limit": PACK_LIMIT,
    "transformers": transformers.__version__, "use_cache": USE_CACHE,
    "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "peak_gpu_gb": round(infer_peak_gb, 3), "reserved_gpu_gb": round(infer_reserved_gb, 3),
    "under_1_5_gb": bool(infer_reserved_gb < 1.5),
    "groups": total_groups, "units": len(units), "groups_split": split_groups, "retranslated_units": retried, "length_ratio_fixes": ratio_fixed,
    "issues": [list(i) for i in issues], "flores_200": flores, "seconds": round(elapsed, 1),
    "per_sample_groups": {k: len(v) for k, v in results.items()},
}
json.dump(report, open(f"{OUT_DIR}/report_task3.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))

shutil.make_archive(f"{WORK}/task3_outputs", "zip", OUT_DIR)
os.chdir(WORK)
display(FileLink("task3_outputs.zip"))
print("Done!")

### Output (`task3_outputs.zip`)
- `english/sample-N.txt`: English translation, one block per group
- `parallel/sample-N.md`: Hindi and English side by side, per group
- `all_translations.csv` / `.json`: everything in one table
- `report_task3.json`: GPU memory, settings, quality flags, FLORES score